# **Задание 2.** Трансформеры

## Импорт библиотек и модулей

In [1]:
# %%capture
# !pip install polars transformers datasets accelerate evaluate peft bitsandbytes

In [2]:
import os
import random

import torch
import transformers as tfm
import datasets as dts
import polars as pl
import pandas as pd
import evaluate
import peft

import sklearn as sl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import warnings
warnings.filterwarnings("ignore")

Зададим значение `seed` и зафиксируем его:

In [3]:
seed_value = 927

random.seed(seed_value)
torch.manual_seed(seed_value)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.mps.is_available() else
    "cpu"
)
print(f"Using device: {device}")

Using device: mps


## Подготовка датасета, токенизация

In [5]:
splits = {'train': 'train.csv', 'validation': 'valid.csv'}
train_df = pl.read_csv('hf://datasets/MonoHime/ru_sentiment_dataset/' + splits['train'])
val_df = pl.read_csv('hf://datasets/MonoHime/ru_sentiment_dataset/' + splits['validation'])

In [6]:
train_df.head()

,text,sentiment
i64,str,i64
21098,""".с.,и спросил его: о Посланни…",1
21099,"""Роднее всех родных Попала я в …",1
21100,"""Непорядочное отношение к своим…",2
21101,"""). Отсутствуют нормативы, Гост…",1
21102,""" У меня машина в рук…",1


In [7]:
MODEL_NAME = "DeepPavlov/distilrubert-tiny-cased-conversational-5k"

In [8]:
sentiment = tfm.pipeline("sentiment-analysis", model=MODEL_NAME)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/distilrubert-tiny-cased-conversational-5k and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


In [9]:
tokenizer = tfm.AutoTokenizer.from_pretrained(MODEL_NAME)
model = tfm.AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/distilrubert-tiny-cased-conversational-5k and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
def tokenize(batch):
    return tokenizer(batch, padding="max_length", truncation=True, max_length=128)

In [11]:
def apply_tokenization(df: pl.dataframe):
    _tokenization = tokenize(df["text"].to_list())
    df_tok = df.with_columns([
        pl.Series("input_ids", _tokenization["input_ids"]),
        pl.Series("attention_mask", _tokenization["attention_mask"])
    ])
    df_tok = df_tok.drop("text")
    return df_tok

In [12]:
train_df_tok = apply_tokenization(train_df)
val_df_tok = apply_tokenization(val_df)

In [14]:
train_df_tok.head(5)

,sentiment,input_ids,attention_mask
i64,i64,list[i64],list[i64]
21098,1,"[2, 24, … 3]","[1, 1, … 1]"
21099,1,"[2, 674, … 3]","[1, 1, … 1]"
21100,2,"[2, 642, … 0]","[1, 1, … 0]"
21101,1,"[2, 56, … 3]","[1, 1, … 1]"
21102,1,"[2, 82, … 3]","[1, 1, … 1]"


In [ ]:
# NOTE: optinal for faster training on data sample (15k samples per class)
train_df_tok_sample = train_df_tok.group_by("sentiment").map_groups(
    lambda x: x.sample(
        n=min(15000, len(x)),
        with_replacement=False,
        shuffle=True,
        seed=42
    )
)

In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    """
    Custom PyTorch Dataset for polars DataFrame.
    """
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Создание экземпляра датасета
train_dataset = SentimentDataset(
    input_ids=torch.tensor(train_df_tok['input_ids'].to_list()),
    attention_mask=torch.tensor(train_df_tok['attention_mask'].to_list()),
    labels=torch.tensor(train_df_tok['sentiment'].to_list())
)
val_dataset = SentimentDataset(
    input_ids=torch.tensor(val_df_tok['input_ids'].to_list()),
    attention_mask=torch.tensor(val_df_tok['attention_mask'].to_list()),
    labels=torch.tensor(val_df_tok['sentiment'].to_list())
)

## Обучение и валидация модели

In [33]:
args = tfm.TrainingArguments(
    output_dir="./output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=1e-5,
    load_best_model_at_end=True,
    fp16=(device == torch.device("cuda")),
    report_to="none"
)

In [34]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="macro")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="macro")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [35]:
trainer = tfm.Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [36]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.301100,0.326585,0.551332,0.363835,0.510907,0.421400
2,0.286600,0.331754,0.551901,0.365274,0.515338,0.422114
3,0.269400,0.321683,0.562328,0.370076,0.515095,0.428861
4,0.264600,0.333155,0.559532,0.369100,0.517753,0.427630


KeyboardInterrupt: 

In [37]:
metrics = trainer.evaluate()
metrics

{'eval_loss': 0.3331545293331146,
 'eval_accuracy': 0.5595317091667457,
 'eval_precision': 0.36909999229375673,
 'eval_recall': 0.5177533975213605,
 'eval_f1': 0.4276301642778506}

## Визуализация результатов

In [40]:
val_dataset_sample = SentimentDataset(
    input_ids=val_dataset.input_ids[:5],
    attention_mask=val_dataset.attention_mask[:5],
    labels=val_dataset.labels[:5]
)

In [45]:
samples = val_dataset_sample
preds = trainer.predict(samples)
labels = np.argmax(preds.predictions, axis=-1)

for text, true_label, pred_label in zip(val_df["text"][:5], preds.label_ids[:5], labels[:5]):
    print(f"Текст: {text}")
    print(f"Реальная метка: {true_label}, Предсказанная: {pred_label}\n")

Текст: Развода на деньги нет Наблюдаюсь в Лайфклиник по беременности, развода на деньги нет, врачи не плохие, по поводу ресепшен соглашусь (путают документы). 
Реальная метка: 1, Предсказанная: 1

Текст: Отель выбрали потому что рядом со стадионом. Отель 4*. Номер большой. Кровать 2-спальная одна. 2 одеяла. Много подушек. Есть зона отдыха. Чайный сет. 2 бутылки по 0,5 л бесплатно каждый день. Много шкафов. Мини-бар. Утюг и гладильная доска, отдельно прибор для глажки брюк. Санузел общий большой. Ванна, лейка съемная. 2 раковины, фен. Халаты, тапочки. Расширенный пакет косметических средств, даже соль для ванны. Интернет быстрый. Так как этаж высокий был, вид из окна на город. Один недостаток в номере 919: он напротив хозлифта и с утра начинает пользоваться им персонал для обслуживания завтрака на 9 этаже. Очень слышно звон посуды и разговоры. Завтрак не включен, стоит 23 евро. Рядом с отелем есть кафе и subway.
Реальная метка: 0, Предсказанная: 0

Текст: Вылечили Гноился с рождения гла